In [6]:
import pandas as pd
import json

def analyze_synergy(target_ally_champ):
    with open('all_matches.json', 'r') as f:
        data = json.load(f)
    
    rows = []
    for m in data:
        for p in m['info']['participants']:
            rows.append({
                'matchId': m['metadata']['matchId'],
                'teamId': p['teamId'],
                'champion': p['championName'],
                'win': p['win']
            })
    df = pd.DataFrame(rows)

    # 1. ターゲット（味方）がいた試合・チームを特定
    target_matches = df[df['champion'] == target_ally_champ][['matchId', 'teamId', 'win']]
    
    # 2. その味方と同じチームにいた「他のキャラ」の勝率を集計
    synergy_list = []
    for _, match in target_matches.iterrows():
        teammates = df[(df['matchId'] == match['matchId']) & 
                       (df['teamId'] == match['teamId']) & 
                       (df['champion'] != target_ally_champ)]
        for _, tm in teammates.iterrows():
            synergy_list.append({'Teammate': tm['champion'], 'Win': 1 if match['win'] else 0})
    
    if not synergy_list:
        return "データ不足です"

    synergy_df = pd.DataFrame(synergy_list)
    # 試合数が少ないキャラを除外（最低2試合以上など）
    result = synergy_df.groupby('Teammate')['Win'].agg(['count', 'mean'])
    return result[result['count'] >= 2].sort_values(by='mean', ascending=False)

# 実行例：味方に「LeeSin」がいる時の勝率ランキング
print(analyze_synergy("LeeSin"))

             count      mean
Teammate                    
Ezreal           3  1.000000
Varus            2  1.000000
KSante           2  1.000000
Jhin             2  1.000000
Pantheon         4  0.750000
Vayne            3  0.666667
Leblanc          3  0.666667
Annie            2  0.500000
Veigar           2  0.500000
Viktor           2  0.500000
Lulu             2  0.500000
Karma            2  0.500000
Swain            2  0.500000
TwistedFate      2  0.500000
AurelionSol      3  0.333333
Bard             4  0.250000
Nautilus         4  0.250000
Kaisa            2  0.000000
Sylas            2  0.000000
Sion             3  0.000000
